## Working note


Update: Get any year's headlines with a list of months and days and save as excel

In [1]:
# Get only headlines with the links for access

# Use year variable to select year, months and days are optional or get all dates

# Note, Issues with .htm extension starts from https://www.resmigazete.gov.tr/eskiler/2000/06/20000627.htm
# From the first issue to https://www.resmigazete.gov.tr/eskiler/2005/01/20050102.htm, inside links direct to footers in page.
# Example: https://www.resmigazete.gov.tr/eskiler/2004/12/20041231.htm#1 etc.
# After this issue, inside links direct to separate pages https://www.resmigazete.gov.tr/eskiler/2005/01/20050102-1.htm etc.



# Until there, the URL style is https://www.resmigazete.gov.tr/arsiv/23894.pdf where the last number indicates Issue number
# The first issue is https://www.resmigazete.gov.tr/arsiv/1.pdf , in Arabic
# The first Turkish issue is https://www.resmigazete.gov.tr/arsiv/1054.pdf 


### Titles of Resmi Gazete

In [ ]:
import time
import re
import unicodedata
import pandas as pd
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from datetime import datetime


class ResmiGazeteScraper:
    def __init__(self, year: int, base_url: str = "https://www.resmigazete.gov.tr/eskiler"):
        self.year = int(year)
        self.base_url = base_url.rstrip("/")
        self.site_root = "https://www.resmigazete.gov.tr/"  # for resolving "local" hrefs robustly

        self.months = [f"{m:02d}" for m in range(1, 13)]
        self.days = [f"{d:02d}" for d in range(1, 32)]
        self.rows = []
        self.df = pd.DataFrame(columns=["Datetime", "Text", "Hyperlink"])

        # Skip any word that starts with "ilan" (ilan, ilanlar, ilanları, ilanın, ilana, ...)
        self.ilan_re = re.compile(r"\bilan\w*", flags=re.IGNORECASE)

        # cleanup map for common “weird” characters / cp1252 artifacts
        self.bad_char_map = {
            "\x96": "-",   # en dash
            "\x97": "-",   # em dash (often shows as )
            "": "-",      # explicit visible artifact
            "\x91": "'", "\x92": "'",  # curly single quotes
            "\x93": '"', "\x94": '"',  # curly double quotes
            "\xa0": " ",   # nbsp
        }

        self.session = requests.Session()
        self.session.headers.update({
            "User-Agent": "Mozilla/5.0 (compatible; ResmiGazeteScraper/1.0)"
        })

    def _valid_date(self, month: str, day: str) -> bool:
        try:
            datetime(self.year, int(month), int(day))
            return True
        except ValueError:
            return False

    def _page_url(self, month: str, day: str) -> str:
        return f"{self.base_url}/{self.year}/{month}/{self.year}{month}{day}.htm"

    def _pick_container(self, soup: BeautifulSoup):
        for css in ["#html-content", "#AutoNumber1", "body"]:
            node = soup.select_one(css)
            if node is not None:
                return css, node
        return "document", soup

    # Turkish-safe normalize (does NOT remove Turkish letters)
    def _normalize_text_tr(self, text: str) -> str:
        if not text:
            return ""

        t = text.replace("\r", " ").replace("\n", " ")

        for bad, good in self.bad_char_map.items():
            t = t.replace(bad, good)

        # Normalize unicode (keeps Turkish chars like ğ, ş, ı, İ)
        t = unicodedata.normalize("NFKC", t)

        # Remove control characters (but keep letters)
        t = "".join(ch for ch in t if unicodedata.category(ch) not in ("Cc", "Cf"))

        # Collapse whitespace
        t = " ".join(t.split()).strip()
        return t

    def _should_skip_text(self, text: str) -> bool:
        # Omit texts containing Æ or Å
        if "Æ" in text or "Å" in text:
            return True

        # Omit ilan / ilanlar / ilanları / ilan... (any case)
        if self.ilan_re.search(text):
            return True

        return False

    # resolve hrefs that are local/relative to a proper absolute URL
    def _resolve_href(self, page_url: str, href: str) -> str:
        href = (href or "").strip()
        if not href:
            return ""

        # If it starts with "/" it should be joined to site root.
        if href.startswith("/"):
            return urljoin(self.site_root, href)

        # If it is already absolute, keep it.
        if href.startswith("http://") or href.startswith("https://"):
            return href

        # Otherwise, try joining with the daily page URL (works for relative paths)
        return urljoin(page_url, href)

    def _parse_links(self, html: str, page_url: str, debug: bool = False):
        soup = BeautifulSoup(html, "html.parser")
        css_used, container = self._pick_container(soup)

        anchors = container.select("a[href]") if hasattr(container, "select") else soup.select("a[href]")

        if debug:
            print(f"  container used: {css_used}, links found: {len(anchors)}")

        out = []
        for a in anchors:
            href_raw = a.get("href") or ""
            resolved = self._resolve_href(page_url, href_raw)
            if not resolved:
                continue

            text_raw = a.get_text(" ", strip=True)
            text = self._normalize_text_tr(text_raw)

            # ✅ NEW: omit text under 3 characters
            if len(text) < 3:
                continue

            if self._should_skip_text(text):
                continue

            out.append((text, resolved))

        return out

    def scrape(self, debug: bool = True):
        self.rows = []

        for month in self.months:
            for day in self.days:
                if not self._valid_date(month, day):
                    continue

                page_url = self._page_url(month, day)

                try:
                    r = self.session.get(page_url, timeout=20)
                    if not r.encoding:
                        r.encoding = "utf-8"
                except requests.RequestException as e:
                    print(f"failed to retrive page {page_url} ({type(e).__name__})")
                    time.sleep(1)
                    continue

                if r.status_code != 200:
                    print(f"failed to retrive page {page_url}")
                    time.sleep(1)
                    continue

                print(f"accessing page {page_url}")

                links = self._parse_links(r.text, page_url, debug=debug)

                if debug and len(links) == 0:
                    print("  NOTE: page returned 200 but no <a href> found in selected container (after filtering).")

                date_str = f"{self.year}-{month}-{day}"
                for text, link in links:
                    self.rows.append({"Datetime": date_str, "Text": text, "Hyperlink": link})

                time.sleep(1)

    def to_dataframe(self):
        self.df = pd.DataFrame(self.rows, columns=["Datetime", "Text", "Hyperlink"])
        return self.df

    def save_to_excel(self, filepath: str):
        if self.df is None or self.df.empty:
            self.to_dataframe()
        self.df.to_excel(filepath, index=False, sheet_name="content")
        return filepath

In [34]:
for year in range(2001, 2002):  # 2001..2025
    scraper = ResmiGazeteScraper(year=year)
    scraper.months = [f"{m:02d}" for m in range(1, 13)]
    scraper.days   = [f"{d:02d}" for d in range(1, 32)]  # keep 1..31; invalid dates are skipped internally

    scraper.scrape(debug=False)
    scraper.to_dataframe()
    scraper.save_to_excel(rf"resmigazete_all\TRofficialgazette_{year}_titles.xlsx")

    print(f"Saved year {year}: {len(scraper.df)} rows")

accessing page https://www.resmigazete.gov.tr/eskiler/2001/01/20010101.htm
accessing page https://www.resmigazete.gov.tr/eskiler/2001/01/20010102.htm
accessing page https://www.resmigazete.gov.tr/eskiler/2001/01/20010103.htm
accessing page https://www.resmigazete.gov.tr/eskiler/2001/01/20010104.htm
accessing page https://www.resmigazete.gov.tr/eskiler/2001/01/20010105.htm
accessing page https://www.resmigazete.gov.tr/eskiler/2001/01/20010106.htm
accessing page https://www.resmigazete.gov.tr/eskiler/2001/01/20010107.htm
accessing page https://www.resmigazete.gov.tr/eskiler/2001/01/20010108.htm
accessing page https://www.resmigazete.gov.tr/eskiler/2001/01/20010109.htm
accessing page https://www.resmigazete.gov.tr/eskiler/2001/01/20010110.htm
accessing page https://www.resmigazete.gov.tr/eskiler/2001/01/20010111.htm
accessing page https://www.resmigazete.gov.tr/eskiler/2001/01/20010112.htm
accessing page https://www.resmigazete.gov.tr/eskiler/2001/01/20010113.htm
accessing page https://ww